In [1]:
import os
import struct
import torch
from torch.utils.data import Dataset, DataLoader

In [2]:
class CustomDataset(Dataset):
    def __init__ (self, root, train = True, transform = None):
        self.root = root
        self.train = train
        self.transform = transform

        self.classes = ["T-shirt/top","Trouser","Pullover","Dress","Coat","Sandal","Shirt","Sneaker","Bag","Ankle boot"]

        if train:
            image_file = "train-images-idx3-ubyte"
            label_file = "train-labels-idx1-ubyte"
        else:
            image_file = "t10k-images-idx3-ubyte"
            label_file = "t10k-labels-idx1-ubyte"

        img_path = os.path.join(root, image_file)
        lable_path = os.path.join(root, label_file)

        self.images = self._load_images(img_path)
        self.labels = self._load_lables(lable_path)

    def _load_images(self, img_path):
        with open(img_path, "rb") as f:
            magic, no_of_sample, rows, cols = struct.unpack(">IIII", f.read(16))
            data = f.read()

        images = torch.tensor(list(data), dtype = torch.uint8)
        images = images.reshape(no_of_sample, rows, cols)

        return images

    def _load_lables(self, lable_path):
        with open(lable_path, "rb") as f:
            magic, no_of_lables = struct.unpack(">II", f.read(8))
            data = f.read()

        lables = torch.tensor(list(data), dtype = torch.long)

        return lables

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = self.images[index]
        lable = self.labels[index]

        if self.transform is not None:
            image = self.transform(image)

        return image, lable

In [3]:
root = './data'

In [4]:
root = os.path.join(root, 'FashionMNIST', 'raw')

In [5]:
root

'./data\\FashionMNIST\\raw'

In [6]:
from torchvision import transforms

In [7]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandAugment(num_ops = 2, magnitude = 9, num_magnitude_bins = 31 ),
    transforms.ToTensor()
])

In [8]:
train_data = CustomDataset(
    root,
    train = True,
    transform = transform
)


test_data = CustomDataset(
    root,
    train = False,
    transform = transform
)

In [9]:
len(train_data), len(test_data)

(60000, 10000)

In [10]:
img, lable = train_data[0]
iii, lll = test_data[0]
print(f'train sample info : {img.shape}, {lable}, {train_data.classes[lable.item()]}')
print(f'test sample info : {iii.shape}, {lll}, {train_data.classes[lll.item()]}')

train sample info : torch.Size([1, 28, 28]), 9, Ankle boot
test sample info : torch.Size([1, 28, 28]), 9, Ankle boot


In [11]:
class MyCNN(torch.nn.Module):
    def __init__(self, in_channels, no_of_kernels, no_of_classes):
        super().__init__()

        self.block1 = torch.nn.Sequential(
            torch.nn.Conv2d(in_channels = in_channels, out_channels = no_of_kernels, kernel_size = 3, stride = 1, padding = 1),
            torch.nn.ReLU(),
            torch.nn.Conv2d(in_channels = no_of_kernels, out_channels = no_of_kernels, kernel_size = 3, stride = 1, padding = 1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size = 2),
            torch.nn.Dropout2d( p = 0.2)
        )

        self.block2 = torch.nn.Sequential(
            torch.nn.Conv2d(in_channels = no_of_kernels, out_channels = no_of_kernels, kernel_size = 3, stride = 1, padding = 1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size = 2),
            torch.nn.Dropout2d( p = 0.2)
        )

        self.classifierblock = torch.nn.Sequential(
            torch.nn.Flatten(),
            torch.nn.Dropout(p = 0.5),
            torch.nn.Linear(in_features = no_of_kernels * 7 * 7, out_features = no_of_classes)
        )

    def forward(self, x):
        return(self.classifierblock(self.block2(self.block1(x))))

In [12]:
from torchmetrics import Accuracy

In [39]:
def training_loop(model: torch.nn.Module, data: torch.utils.data.DataLoader, loss_fn: torch.nn.Module, optimizer: torch.optim.Optimizer, accuracy: Accuracy, device: torch.device):
    model = model.to(device)
    accuracy = accuracy.to(device)

    model.train()

    accuracy.reset()
    train_loss = 0

    for X, y in data:
        X, y = X.to(device), y.to(device)

        y_pred = model(X)

        loss = loss_fn(y_pred, y)
        train_loss += loss.item()
        accuracy.update(y_pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    acc = accuracy.compute()
    train_loss /= len(data)
    print(f'\ttrain loss: \t{train_loss:.4f} \t||\t train acc: \t{acc:.4f}')

    return train_loss, acc

In [40]:
def testing_loop(model: torch.nn.Module, data: torch.utils.data.DataLoader, loss_fn: torch.nn.Module, accuracy: Accuracy, device: torch.device):
    model = model.to(device)
    accuracy = accuracy.to(device)

    model.eval()

    accuracy.reset()
    test_loss = 0

    with torch.inference_mode():
        for X, y in data:
            X, y = X.to(device), y.to(device)

            y_pred = model(X)
            
            loss = loss_fn(y_pred, y)
            test_loss += loss.item()
            accuracy.update(y_pred, y)
  
    acc = accuracy.compute()
    test_loss /= len(data)
    print(f'\ttest loss: \t{test_loss:.4f} \t||\t test acc: \t{acc:.4f}')

    return test_loss, acc

In [15]:
device  = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [16]:
train_dataloader = DataLoader(
    train_data,
    batch_size = 64,
    shuffle = True
)

test_dataloader = DataLoader(
    test_data,
    batch_size = 100
)

In [17]:
model = MyCNN(in_channels = 1, no_of_kernels = 10, no_of_classes = len(test_data.classes)).to(device)

In [18]:
next(model.parameters()).device

device(type='cuda', index=0)

In [19]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params = model.parameters(), lr = 0.01, momentum = 0.9)
accuracy = Accuracy(task = 'multiclass', num_classes = len(test_data.classes))

In [20]:
torch.manual_seed(42)

epoch = 100

for ep in range(epoch):
    print(f'\nepoch {ep}:')

    training_loop(model, train_dataloader, loss_fn, optimizer, accuracy, device)
    testing_loop(model, test_dataloader, loss_fn, accuracy, device)


epoch 0:
	train loss: 	1.0987 	||	 train acc: 	0.5913
	test loss: 	0.6897 	||	 test acc: 	0.7450

epoch 1:
	train loss: 	0.7917 	||	 train acc: 	0.7076
	test loss: 	0.6276 	||	 test acc: 	0.7700

epoch 2:
	train loss: 	0.7355 	||	 train acc: 	0.7299
	test loss: 	0.6063 	||	 test acc: 	0.7841

epoch 3:
	train loss: 	0.7123 	||	 train acc: 	0.7377
	test loss: 	0.5714 	||	 test acc: 	0.7848

epoch 4:
	train loss: 	0.6932 	||	 train acc: 	0.7460
	test loss: 	0.5519 	||	 test acc: 	0.7958

epoch 5:
	train loss: 	0.6875 	||	 train acc: 	0.7473
	test loss: 	0.5596 	||	 test acc: 	0.7954

epoch 6:
	train loss: 	0.6740 	||	 train acc: 	0.7523
	test loss: 	0.5597 	||	 test acc: 	0.7930

epoch 7:
	train loss: 	0.6680 	||	 train acc: 	0.7555
	test loss: 	0.5387 	||	 test acc: 	0.8003

epoch 8:
	train loss: 	0.6615 	||	 train acc: 	0.7557
	test loss: 	0.5293 	||	 test acc: 	0.8051

epoch 9:
	train loss: 	0.6553 	||	 train acc: 	0.7593
	test loss: 	0.5348 	||	 test acc: 	0.8076

epoch 10:
	train lo

In [21]:
from pathlib import Path

In [22]:
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok = True)

MODEL_NAME = "cnn_wth_dropouts_model.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

print(f"Saving model to: {MODEL_SAVE_PATH}")
torch.save(obj=model.state_dict(), f=MODEL_SAVE_PATH)

Saving model to: models\cnn_wth_dropouts_model.pth


In [23]:
CT_model = MyCNN(1, 10, len(test_data.classes))

In [24]:
m_path = os.path.join(MODEL_PATH, MODEL_NAME)
m_path

'models\\cnn_wth_dropouts_model.pth'

In [25]:
CT_model.load_state_dict(torch.load(f = m_path))

<All keys matched successfully>

In [26]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(params = CT_model.parameters(), lr = 0.01, momentum = 0.9)
accuracy = Accuracy(task = 'multiclass', num_classes = len(test_data.classes))

In [28]:
torch.manual_seed(42)

epoch = 1000

for ep in range(epoch):
    print(f'\nepoch {ep}:')

    training_loop(CT_model, train_dataloader, loss_fn, optimizer, accuracy, device)
    testing_loop(CT_model, test_dataloader, loss_fn, accuracy, device)


epoch 0:
	train loss: 	0.5814 	||	 train acc: 	0.7878
	test loss: 	0.4616 	||	 test acc: 	0.8278

epoch 1:
	train loss: 	0.5866 	||	 train acc: 	0.7854
	test loss: 	0.4529 	||	 test acc: 	0.8398

epoch 2:
	train loss: 	0.5794 	||	 train acc: 	0.7877
	test loss: 	0.4574 	||	 test acc: 	0.8373

epoch 3:
	train loss: 	0.5808 	||	 train acc: 	0.7878
	test loss: 	0.4488 	||	 test acc: 	0.8344

epoch 4:
	train loss: 	0.5813 	||	 train acc: 	0.7862
	test loss: 	0.4513 	||	 test acc: 	0.8362

epoch 5:
	train loss: 	0.5855 	||	 train acc: 	0.7857
	test loss: 	0.4498 	||	 test acc: 	0.8402

epoch 6:
	train loss: 	0.5835 	||	 train acc: 	0.7847
	test loss: 	0.4584 	||	 test acc: 	0.8376

epoch 7:
	train loss: 	0.5836 	||	 train acc: 	0.7870
	test loss: 	0.4479 	||	 test acc: 	0.8354

epoch 8:
	train loss: 	0.5823 	||	 train acc: 	0.7851
	test loss: 	0.4514 	||	 test acc: 	0.8367

epoch 9:
	train loss: 	0.5848 	||	 train acc: 	0.7875
	test loss: 	0.4494 	||	 test acc: 	0.8362

epoch 10:
	train lo

In [29]:
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok = True)

MODEL_NAME = "cnn_1k_model.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

print(f"Saving model to: {MODEL_SAVE_PATH}")
torch.save(obj=CT_model.state_dict(), f=MODEL_SAVE_PATH)

Saving model to: models\cnn_1k_model.pth


acc_score and loss are almost same even after 1K training ( most probably underfitting):

    why?:

    1. model is to shallow
    2. regularizations (dropouts) are too aggrisive [can tell cuz test > train]
    3. constant lr of 0.01 is too aggrisive. need to introduce a decaying lr.
    

In [30]:
class BetterCNN(torch.nn.Module):
    def __init__(self, input_channels, no_of_class):
        super().__init__()

        self.layers = torch.nn.Sequential(

            torch.nn.Conv2d(input_channels, 32, kernel_size = 3, stride = 1, padding = 1),
            torch.nn.BatchNorm2d(32),
            torch.nn.ReLU(),

            torch.nn.Conv2d(32, 32, kernel_size = 3, stride = 1, padding = 1),
            torch.nn.BatchNorm2d(32),
            torch.nn.ReLU(),

            torch.nn.MaxPool2d(kernel_size = 2),
            torch.nn.Dropout2d(p = 0.1),

            torch.nn.Conv2d(32, 64, kernel_size = 3, padding = 1),
            torch.nn.BatchNorm2d(64),
            torch.nn.ReLU(),

            torch.nn.Conv2d(64, 64, kernel_size = 3, padding = 1),
            torch.nn.BatchNorm2d(64),
            torch.nn.ReLU(),

            torch.nn.MaxPool2d(2),
            torch.nn.Dropout2d(p = 0.1),

            torch.nn.Conv2d(64, 128, kernel_size = 3, padding = 1),
            torch.nn.BatchNorm2d(128),
            torch.nn.ReLU()
        )

        self.classifier = torch.nn.Sequential(
            torch.nn.Flatten(),
            torch.nn.Linear(128*7*7, 128),
            torch.nn.ReLU(),
            
            torch.nn.Dropout(p = 0.2),

            torch.nn.Linear(128, no_of_class)
        )

    def forward(self, x):
        return self.classifier(self.layers(x))

In [33]:
def save_model(model: torch.nn.Module, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), path)

In [34]:
better_model = BetterCNN(1, len(test_data.classes))

In [35]:
better_model

BetterCNN(
  (layers): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Dropout2d(p=0.1, inplace=False)
    (8): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU()
    (11): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (13): ReLU()
    (14): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_

In [37]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(params = better_model.parameters(), lr = 0.01, weight_decay = 1e-2)
accurac = Accuracy(task = 'multiclass', num_classes = len(test_data.classes))
schedular = torch.optim.lr_scheduler.StepLR(optimizer, step_size = 5, gamma = 0.9)

In [41]:
torch.manual_seed(42)

MODEL_PATH = Path("models")
log_dir = Path("logs")

MODEL_PATH.mkdir(parents=True, exist_ok = True)
log_dir.mkdir(parents = True, exist_ok = True)

MODEL_NAME = "better_cnn_model.pth"
log_name = "log.txt"

save_path = MODEL_PATH / MODEL_NAME
log_path = log_dir / log_name

epoch = 100
best_acc = 0.0000

for ep in range(epoch):
    print(f'\nepoch {ep}:')

    train_loss, train_acc = training_loop(better_model, train_dataloader, loss_fn, optimizer, accuracy, device)
    test_loss, test_acc = testing_loop(better_model, test_dataloader, loss_fn, accuracy, device)

    if test_acc > best_acc:
        best_acc = test_acc
        save_model(better_model, save_path)
        with open(log_path, 'a') as f:
            f.write(f'epoch: {ep + 1} \tbest test acc: {best_acc}, \tmodels state dict saved!\n')

    schedular.step()


epoch 0:
	train loss: 	0.5951 	||	 train acc: 	0.7781
	test loss: 	0.4458 	||	 test acc: 	0.8321

epoch 1:
	train loss: 	0.5028 	||	 train acc: 	0.8165
	test loss: 	0.4155 	||	 test acc: 	0.8518

epoch 2:
	train loss: 	0.4582 	||	 train acc: 	0.8358
	test loss: 	0.3937 	||	 test acc: 	0.8531

epoch 3:
	train loss: 	0.4337 	||	 train acc: 	0.8439
	test loss: 	0.3530 	||	 test acc: 	0.8680

epoch 4:
	train loss: 	0.4112 	||	 train acc: 	0.8512
	test loss: 	0.3819 	||	 test acc: 	0.8659

epoch 5:
	train loss: 	0.3901 	||	 train acc: 	0.8625
	test loss: 	0.3305 	||	 test acc: 	0.8790

epoch 6:
	train loss: 	0.3771 	||	 train acc: 	0.8667
	test loss: 	0.3522 	||	 test acc: 	0.8720

epoch 7:
	train loss: 	0.3713 	||	 train acc: 	0.8669
	test loss: 	0.3269 	||	 test acc: 	0.8771

epoch 8:
	train loss: 	0.3607 	||	 train acc: 	0.8704
	test loss: 	0.3108 	||	 test acc: 	0.8876

epoch 9:
	train loss: 	0.3546 	||	 train acc: 	0.8730
	test loss: 	0.3112 	||	 test acc: 	0.8913

epoch 10:
	train lo

In [42]:
better_model.state_dict()

OrderedDict([('layers.0.weight',
              tensor([[[[ 7.9689e-03,  2.3005e-01, -3.6872e-02],
                        [ 5.8185e-01, -7.2924e-01, -1.9192e-01],
                        [ 1.2310e-01, -2.0038e-01,  2.3141e-01]]],
              
              
                      [[[ 3.7544e-03,  2.6666e-03,  2.7915e-03],
                        [-4.6846e-03,  1.7615e-03,  1.3579e-02],
                        [ 3.2508e-03, -5.0764e-04,  4.6137e-03]]],
              
              
                      [[[-2.7176e-02,  5.1946e-02,  8.3623e-02],
                        [ 4.1254e-03, -1.5422e-02, -8.1016e-02],
                        [ 1.0816e-01, -1.2827e+00,  1.8465e-02]]],
              
              
                      [[[-1.8811e-01, -5.3394e-01, -7.1313e-02],
                        [-4.6298e-01, -1.4717e-01,  2.5347e-02],
                        [-6.9513e-02,  4.3430e-02, -2.4505e-02]]],
              
              
                      [[[-1.4625e-01, -2.4641e-01, -5.8328e